<h1 style="text-align: center;">Cafe Sales Analysis: Dirty Dataset Cleaning</h1>
<!-- <h3 style="font-style: italic; text-align: center;">Dirty Dataset Cleaning</h3> -->

The purpose of this notebook is to clean and standardize a realistic but imperfect cafe sales dataset so it can be used for trustworthy exploratory analysis and business insights. By detecting missing values, invalid entries, and inconsistent transaction details, it walks through practical data-wrangling techniques to prepare the data for accurate reporting, trend analysis, and decision-making.

In [1]:
import numpy as np
import pandas as pd
from data_cleaning_utils import MAX_CHARS, RANDOM_STATE, RAW_DATA_PATH, analyze_metadata

---

## Data Description

> The Dirty Cafe Sales dataset contains 10,000 rows and 8 columns of synthetic data representing sales transactions in a cafe. This dataset is intentionally "dirty", with missing values, inconsistent data, and errors introduced to provide a realistic scenario for data cleaning and exploratory data analysis (EDA). It can be used—especially as the main focus of this notebook—to practice cleaning techniques and data wrangling.

*Adapted from [Cafe Sales - Dirty Data for Cleaning Training](https://www.kaggle.com/datasets/ahmedmohamed2003/cafe-sales-dirty-data-for-cleaning-training) by [Ahmed Mohamed](https://www.kaggle.com/ahmedmohamed2003).*

### File Information

* **File Name:** `dirty_cafe_sales.csv`
* **Number of Rows:** 10,000
* **Number of Columns:** 8

### Columns Description

| Column Name | Description | Example Values |
| --- | --- | --- |
| `Transaction ID` | A unique identifier for each transaction. Always present and unique. | `TXN_1234567` |
| `Item` | The name of the item purchased. May contain missing or invalid values (e.g., "ERROR"). | `Coffee, Sandwich` |
| `Quantity` | The quantity of the item purchased. May contain missing or invalid values. | `1, 3, UNKNOWN` |
| `Price Per Unit` | The price of a single unit of the item. May contain missing or invalid values. | `2.00, 4.00` |
| `Total Spent` | The total amount spent on the transaction. Calculated as `Quantity * Price Per Unit`. | `8.00, 12.00` |
| `Payment Method` | The method of payment used. May contain missing or invalid values (e.g., None, "UNKNOWN"). | `Cash, Credit Card` |
| `Location` | The location where the transaction occurred. May contain missing or invalid values. | `In-store, Takeaway` |
| `Transaction Date` | The date of the transaction. May contain missing or incorrect values. | `2023-01-01` |

### Data Characteristics

1. **Missing Values**: Some columns (e.g., Item, Payment Method, Location) may contain missing values represented as None or empty cells.
2. **Invalid Values**: Some rows contain invalid entries like "ERROR" or "UNKNOWN" to simulate real-world data issues.
3. **Price Consistency**: Prices for menu items are consistent but may have missing or incorrect values introduced.
* The dataset includes the following menu items with their respective price ranges:

    | Item | Price ($) |
    | --- | :---: |
    | Coffee | 2 |
    | Tea | 1.5 |
    | Sandwich | 4 |
    | Salad | 5 |
    | Cake | 3 |
    | Cookie | 1 |
    | Smoothie | 4 |
    | Juice | 3 |  

---

## Metadata Analysis

In [2]:
# read data file and display a sample of 5 rows
raw_data = pd.read_csv(RAW_DATA_PATH)
raw_data.head()

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,ERROR,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,UNKNOWN,UNKNOWN,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11


In [3]:
# perform metadata analysis on raw data
with pd.option_context("display.max_colwidth", MAX_CHARS):
    display(analyze_metadata(raw_data))

,Columns,Non-Null Count,Dtype,Unique Values
0,Transaction ID,10000,str,"[TXN_1961373, TXN_4977031, TXN_4271903, TXN_7034554, TXN_3160411, TXN_2602893, TXN_4433211, TXN_6699534, TXN_4717867..."
1,Item,9667,str,"[Coffee, Cake, Cookie, Salad, Smoothie, UNKNOWN, Sandwich, nan, ERROR, Juice, Tea]"
2,Quantity,9862,str,"[2, 4, 5, 3, 1, ERROR, UNKNOWN, nan]"
3,Price Per Unit,9821,str,"[2.0, 3.0, 1.0, 5.0, 4.0, 1.5, nan, ERROR, UNKNOWN]"
4,Total Spent,9827,str,"[4.0, 12.0, ERROR, 10.0, 20.0, 9.0, 16.0, 15.0, 25.0, 8.0, 5.0, 3.0, 6.0, nan, UNKNOWN, 2.0, 1.0, 7.5, 4.5, 1.5]"
5,Payment Method,7421,str,"[Credit Card, Cash, UNKNOWN, Digital Wallet, ERROR, nan]"
6,Location,6735,str,"[Takeaway, In-store, UNKNOWN, nan, ERROR]"
7,Transaction Date,9841,str,"[2023-09-08, 2023-05-16, 2023-07-19, 2023-04-27, 2023-06-11, 2023-03-31, 2023-10-06, 2023-10-28, 2023-07-28, 2023-12..."


<table>
  <tr>
    <th style="width:10rem">Category</th>
    <th>Insight</th>
  </tr>
  <tr>
    <td>Data Dimensionality</td>
    <td>The number of <code>Columns</code>, and also <code>Non-Null Count</code> for <code>Transaction ID</code>, matches the excerpt from <a href="#file-information">File Information</a> section: <code>8</code> variables across <code>10000</code> unique observations.</td>
  </tr>
  <tr>
    <td><b>Missing Data</b></td>
    <td><b>Missing values are identified from <code>Non-Null Count</code> values across all <code>Columns</code> except <code>Transaction ID</code>?represented by <code>nan</code> in <code>Unique Values</code>?with varying counts.</b></td>
  </tr>
  <tr>
    <td rowspan="3"><b>Invalid Data Type</b></td>
    <td><b><code>Quantity</code>, representing the number of purchased menu items, should be of type <code>int</code>.</b></td>
  </tr>
  <tr>
    <td><b><code>Price Per Unit</code> for menu items, one of which includes a decimal point (<code>1.5</code>), should be of type <code>float</code>. The same applies to <code>Total Spent</code>, which is calculated as <code>Quantity (int)</code> multiplied by <code>Price Per Unit (float)</code>.</b></td>
  </tr>
  <tr>
    <td><b><code>Transaction Date</code> will be converted to the appropriate <code>datetime</code> type.</b></td>
  </tr>
  <tr>
    <td><b>Invalid Values</b></td>
    <td><b>Invalid entries detected on several <code>Columns</code>, such as <code>ERROR & UNKNOWN</code> in <code>Unique Values</code>.</b></td>
  </tr>
</table>

---

## Data Cleaning

>To clean this dataset, consider the following steps:
>
>1. Handle Missing Values:
>    * Fill missing numeric values with the median or mean.
>    * Replace missing categorical values with the mode or "Unknown."
>
>2. Handle Invalid Values:
>    * Replace invalid entries like "ERROR" and "UNKNOWN" with NaN or appropriate values.
>
>3. Date Consistency:
>    * Ensure all dates are in a consistent format.
>    * Fill missing dates with plausible values based on nearby records.
>
>4. Feature Engineering:
>    * Create new columns, such as Day of the Week or Transaction Month, for further analysis.

### Invalid Values

* Following on suggested data cleaning processes, invalid entries may be replaced with `nan` or the appropriate values.
    * This step should therefore come first before imputing missing values.
* Starting from invalid values on the first column `Item`, previous section on [Data Characteristics](#data-characteristics) provides information regarding a list of menu items which uniquely corresponds to their `Price Per Unit`.
    * For example, `coffee` is the only one on the list with the `Price ($)` of `2.0`.
        * Therefore, replacing invalid values in these columns can be done by maintaining a kind of "consistency" between them. 
        * E.g. `ERROR` entries in `Item` with `Price Per Unit` of `2.0` is certain to be `Coffee`, and vice versa.
    * With the exceptions of `Sandwich` which shares the same `Price Per Unit` of `4.0` with `Smoothie`, and `Cake` with `Juice (3.0)`.
        * The replacement strategy may only be implemented in "one-way".
        * I.e. `Sandwich & Smoothie` have `Price Per Unit = 4.0`, but not all observations with `Price Per Unit = 4.0` is either `Sandwich / Smoothie`.
* Next, invalid entries in the form of `UNKNOWN / ERROR` are replaced with the missing value `nan` so they can then be imputed.

In [4]:
# normalize invalid entries before filling gaps
price_to_item = {
    "2.0": "Coffee",
    "1.5": "Tea",
    "5.0": "Salad",
    "1.0": "Cookie",
}
item_to_price = {
    "Coffee": "2.0",
    "Tea": "1.5",
    "Sandwich": "4.0",
    "Salad": "5.0",
    "Cake": "3.0",
    "Cookie": "1.0",
    "Smoothie": "4.0",
    "Juice": "3.0",
}

cleaned_data = raw_data.copy()
for price, item in price_to_item.items():
    mask = (
        (cleaned_data["Price Per Unit"] == price)
        & (cleaned_data["Item"].isna() | cleaned_data["Item"].isin(["ERROR", "UNKNOWN"]))
    )
    cleaned_data.loc[mask, "Item"] = item

for item, price in item_to_price.items():
    mask = (
        (cleaned_data["Item"] == item)
        & (cleaned_data["Price Per Unit"].isna() | cleaned_data["Price Per Unit"].isin(["ERROR", "UNKNOWN"]))
    )
    cleaned_data.loc[mask, "Price Per Unit"] = price

for column in cleaned_data.columns:
    cleaned_data.loc[cleaned_data[column].isin(["UNKNOWN", "ERROR"]), column] = np.nan

### Invalid Data Type

Conversion process to the appropriate data types is also performed to make imputation easier based on the type of each variable:
* Mean/median for numerical data types and mode for categorical data types.
* Additionally, `category` data type is used to better represent the categorical variable—even more so when the `Item` takes on a limited and/or fixed number of possible values as stated in [Data Characteristics](#data-characteristics) section.

In [5]:
# convert columns to the appropriate data types
categorical_columns = ["Item", "Payment Method", "Location"]
numerical_columns_dtypes = {
    "Quantity": "Int64",
    "Price Per Unit": "Float64",
    "Total Spent": "Float64",
}

for column in categorical_columns:
    cleaned_data[column] = cleaned_data[column].astype("category")

for column, dtype in numerical_columns_dtypes.items():
    cleaned_data[column] = cleaned_data[column].astype(dtype)

### Date Consistency

Imputation begins with the last column in the dataset, `Transaction Date`:
* This also implements the recommended data-cleaning practice of maintaining consistency between values and variable formatting.
* To ensure all dates are in similar formatting, `pd.to_datetime()` is used to convert the column to `datetime` data type which follows the defined format.

In [6]:
# convert "Transaction Date" column to datetime format
cleaned_data["Transaction Date"] = pd.to_datetime(cleaned_data["Transaction Date"], format="%Y-%m-%d")

Next, regarding the suggestion to `fill missing dates with plausible values based on nearby records`:
* Meaning those records need to be inspected by slicing a sample of observations with the missing date together with a few surrounding instances,
* To see whether there is a pattern or similarity that can be used to impute `nan` values in the variable.

In [7]:
# inspect rows around the first missing transaction date
first_missing_index = cleaned_data.index[cleaned_data["Transaction Date"].isna()][0]
cleaned_data.iloc[first_missing_index - 3 : first_missing_index + 4]

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
8,TXN_4717867,NaN,5,3.0,15.0,NaN,Takeaway,2023-07-28
9,TXN_2064365,Sandwich,5,4.0,20.0,NaN,In-store,2023-12-31
10,TXN_2548360,Salad,5,5.0,25.0,Cash,Takeaway,2023-11-07
11,TXN_3051279,Sandwich,2,4.0,8.0,Credit Card,Takeaway,NaT
12,TXN_7619095,Sandwich,2,4.0,8.0,Cash,In-store,2023-05-03
13,TXN_9437049,Cookie,5,1.0,5.0,NaN,Takeaway,2023-06-01
14,TXN_8915701,Tea,2,1.5,3.0,NaN,In-store,2023-03-21


Based on the data selection above:
* Record with the missing `NaT` value in `Transaction Date` does not resemble other nearby records.
    * That is, based on variables such as `Transaction ID`, `Payment Method`, and/or `Location`, the record with the missing value does not form a clear pattern with the neighboring instances.
    * Therefore, it may not be possible to determine what plausible values to impute for the `NaT` data.
    * In other words, it is possible that these missing entries are unrelated to nearby records and occur entirely at random.
* Thus, imputation for these missing values can also be performed randomly based on the available `Transaction ID` data.
    * *To avoid assuming that transaction date falls within a certain range.*

In [8]:
# fill missing transaction dates with random samples from valid dates
valid_dates = cleaned_data["Transaction Date"].dropna()
missing_mask = cleaned_data["Transaction Date"].isna()

cleaned_data.loc[missing_mask, "Transaction Date"] = valid_dates.sample(
    missing_mask.sum(),
    replace=True,
    random_state=RANDOM_STATE,
).to_numpy()

### Missing Values

Imputation is then applied based on the data type and attributes of the other columns:
* Missing categorical values are replaced with the mode, and missing numeric values in `Quantity & Price Per Unit` are filled with the median value.
    * To preserve and give a better sense of the most common value in each column; and
    * To maintain data validity—handling missing entries with proper data rather than using placeholder `UNKNOWN / ERROR` values.
    * *Ultimately, this is a synthetic dataset that will be used to simulate analysis through a dashboard implementation, so any imputation process is applied without assuming any premise beyond what is stated or shown in this notebook.*
* For missing values in `Total Spent`, they are filled according to the defined calculation of `Quantity * Price Per Unit`.

In [9]:
# impute missing values based on the data type and expected values
for categorical_column in categorical_columns:
    mode_value = cleaned_data[categorical_column].mode(dropna=True)
    if not mode_value.empty:
        cleaned_data[categorical_column] = cleaned_data[categorical_column].fillna(mode_value.iloc[0])

for numerical_column in ["Quantity", "Price Per Unit"]:
    median_value = cleaned_data[numerical_column].median()
    cleaned_data[numerical_column] = cleaned_data[numerical_column].fillna(median_value)

cleaned_data["Total Spent"] = cleaned_data["Quantity"] * cleaned_data["Price Per Unit"]

---

## Cleaning Summary

In [10]:
# perform metadata analysis on cleaned data
with pd.option_context("display.max_colwidth", MAX_CHARS):
    display(analyze_metadata(cleaned_data))

,Columns,Non-Null Count,Dtype,Unique Values
0,Transaction ID,10000,str,"[TXN_1961373, TXN_4977031, TXN_4271903, TXN_7034554, TXN_3160411, TXN_2602893, TXN_4433211, TXN_6699534, TXN_4717867..."
1,Item,10000,category,"['Coffee', 'Cake', 'Cookie', 'Salad', 'Smoothie', 'Sandwich', 'Tea', 'Juice'] Categories (8, str): ['Cake', 'Coffee'..."
2,Quantity,10000,Int64,"[2, 4, 5, 3, 1]"
3,Price Per Unit,10000,Float64,"[2.0, 3.0, 1.0, 5.0, 4.0, 1.5]"
4,Total Spent,10000,Float64,"[4.0, 12.0, 10.0, 20.0, 9.0, 16.0, 15.0, 25.0, 8.0, 5.0, 3.0, 6.0, 2.0, 1.0, 7.5, 4.5, 1.5]"
5,Payment Method,10000,category,"['Credit Card', 'Cash', 'Digital Wallet'] Categories (3, str): ['Cash', 'Credit Card', 'Digital Wallet']"
6,Location,10000,category,"['Takeaway', 'In-store'] Categories (2, str): ['In-store', 'Takeaway']"
7,Transaction Date,10000,datetime64[us],"[2023-09-08 00:00:00, 2023-05-16 00:00:00, 2023-07-19 00:00:00, 2023-04-27 00:00:00, 2023-06-11 00:00:00, 2023-03-31..."


<table>
  <tr>
    <th style="width:10rem">Category</th>
    <th>Insight</th>
  </tr>
  <tr>
    <td>Missing Data</td>
    <td><code>Non-Null Count</code> across all <code>Columns</code> is identical, indicating that missing values have been imputed according to the procedure defined in the <a href=#data-cleaning>Data Cleaning</a> section.</td>
  </tr>
  <tr>
    <td rowspan="3">Invalid Data Type</td>
    <td><code>Quantity</code>, representing the number of purchased menu items, has been converted to the <code>Int64</code> type.</td>
  </tr>
  <tr>
    <td><code>Price Per Unit</code> and <code>Total Spent</code> are modified to <code>Float64</code> data type.</td>
  </tr>
  <tr>
    <td><code>Transaction Date</code> is also converted to the appropriate <code>datetime64</code> format.</td>
  </tr>
  <tr>
    <td>Invalid Values</td>
    <td>Invalid entries are corrected and no longer shown in <code>Unique Values</code>.</td>
  </tr>
</table>

In [11]:
# setup filesystem path for cleaned dataset
CLEANED_DATA_DIR = RAW_DATA_PATH.parent.parent / "transformed"
CLEANED_DATA_DIR.mkdir(parents=True, exist_ok=True)
CLEANED_DATA_PATH = CLEANED_DATA_DIR / "clean_cafe_sales.csv"

# write cleaned dataset to .csv file for persistence purposes
cleaned_data.to_csv(CLEANED_DATA_PATH, index=False)
print("CSV export successful")

CSV export successful


---